# Copernicus Marine Service: Access Insitu Data 
* Need to have an account on Copernicus Marine
* The ~/.netrc file contains the account credentials: your login and password for [Copernicus Marine](https://marine.copernicus.eu/)
```
$ more ~/.netrc
machine auth.marine.copernicus.eu
  login xxxxxxxxxx 
  password xxxxxxxxxxxxx
```

In [1]:
import copernicusmarine
import xarray as xr
import hvplot.xarray
import pandas as pd
from datetime import datetime, timezone, timedelta

In [2]:
copernicusmarine.__version__

'2.3.0'

In [3]:

var = 'SLEV'
ndays = 10   # get only last ndays of data

In [4]:
# Get current UTC time
stop_time = datetime.now(timezone.utc)

# Subtract ndays
start_time = stop_time - timedelta(days=ndays)
print(f"start_time: {start_time}, stop_time={stop_time}")

start_time: 2026-04-11 12:56:42.050853+00:00, stop_time=2026-04-21 12:56:42.050853+00:00


## Find stations that have recent data

In [5]:
%%script false --no-raise-error
# comment out the above to run; this just prevents inadvertant running since this takes a while
df_all = copernicusmarine.read_dataframe(
    dataset_id="cmems_obs-ins_med_phybgcwav_mynrt_na_irr",
    dataset_part="latest",
    variables=[var],
    start_datetime=stop_time - timedelta(hours=12),
    end_datetime=stop_time,
)
print(df_all['platform_id'].unique())

## Load data from a specific station

In [6]:
sta = 'Taranto1TG'

In [7]:
df = copernicusmarine.read_dataframe(
    dataset_id="cmems_obs-ins_med_phybgcwav_mynrt_na_irr",
    dataset_part="latest",
    variables=[var],
    start_datetime=start_time,
    end_datetime=stop_time,
    platform_ids=[sta]
)

INFO - 2026-04-21T12:58:02Z - Downloading Copernicus Marine data requires a Copernicus Marine username and password, sign up for free at: https://data.marine.copernicus.eu/register


Copernicus Marine username:

  pendaoriana47@gmail.com


Copernicus Marine password:

  ········


INFO - 2026-04-21T12:58:31Z - Selected dataset version: "202311"
INFO - 2026-04-21T12:58:31Z - Selected dataset part: "latest"
WARNING - 2026-04-21T12:58:31Z - Some of your subset selection [2026-04-11 12:56:42.050853+00:00, 2026-04-21 12:56:42.050853+00:00] for the time dimension exceed the dataset coordinates [2026-03-22 00:00:00+00:00, 2026-04-21 09:15:00+00:00]


In [8]:
df['time'] = pd.to_datetime(df['time'])  # ensure it's datetime64[ns]

In [9]:
ds = df.to_xarray()

In [ ]:
ds

In [10]:
ds = df.set_index('time').to_xarray()
var = str(ds['variable'][0].data)
ds = ds.rename({'value':var})

In [11]:
ds[var].hvplot(x='time', grid=True, title=f'Station {sta}')

:Curve   [time]   (SLEV)